# Python Data Analysis Practice · Delivery Operations

An online retailer receives delivery updates from several carriers. A case can appear more than once as new updates arrive. The operations lead wants to know which cases are still active, which delays need attention, and what to include in a daily briefing.

Download [delivery_exceptions.csv](data/delivery_exceptions.csv) and run the setup cell. Run Jupyter from the notebook's folder so the CSV is under `data/`. In Colab, upload the CSV through the **Files** panel and update the path as shown in the setup comment. Use Python 3.11 or newer. If packages are missing, run `%pip install numpy pandas matplotlib seaborn requests` in a notebook cell.

Write your answer in the blank cell under each question. You can reuse your work on later questions. When a question refers to **cleaned cases**, use normalized text, valid delay values, and the latest update per case. An **active** case has status `open` or `in_progress`.

Refer to [Python Tools](01_python_tools.ipynb) and the [NumPy, pandas and chart sections](02_data_and_llms.ipynb) for class examples.

## Load the CSV

| Column | What it means |
| --- | --- |
| `case_id`, `order_id` | Delivery exception and associated order. A case can have multiple updates. |
| `opened_at`, `updated_at` | When the case was opened and when this update was recorded. |
| `region`, `city`, `carrier` | Where the order was going and who was handling it. |
| `issue_type` | Reason for the case, such as `delayed`, `damaged`, or `missing_parcel`. |
| `status`, `priority` | Current state at the time of the update and its assigned priority. |
| `delay_minutes` | Reported minutes past the delivery promise. Zero is valid. Blanks, labels such as `pending sync`, and negative values are unknown for analysis. |
| `customer_message` | Short example of what the customer reported. |

The setup loads the raw export so you can explore it in the questions below.

In [125]:
import pandas as pd
import numpy as np
from pathlib import Path

# Works from the repository root or the Assignments_Solutions notebook folder.
# In Colab, upload the CSV and use Path("delivery_exceptions.csv").
csv_candidates = [
    Path("data/delivery_exceptions.csv"),
    Path("logicmojo_course/03-python-tools-and-data/data/delivery_exceptions.csv"),
    Path("../logicmojo_course/03-python-tools-and-data/data/delivery_exceptions.csv"),
]
csv_path = next((path for path in csv_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find delivery_exceptions.csv; check the CSV location or upload it in Colab.")

raw_cases = pd.read_csv(csv_path)
display(raw_cases.head())

,case_id,order_id,opened_at,updated_at,region,city,carrier,issue_type,status,priority,delay_minutes,customer_message
0,7011,ORD-8411,2026-10-06T13:17,2026-10-06T18:57,north,Delhi,ParcelBridge,missing_parcel,resolved,high,150,The parcel has had no scan since it left the s...
1,7027,ORD-8427,2026-10-09T10:44,2026-10-09T14:19,east,Kolkata,ParcelBridge,damaged,resolved,low,25,"The outer box arrived crushed, and one item wa..."
2,7033,ORD-8433,2026-10-10T10:31,2026-10-10T18:17,west,Pune,MetroDispatch,address_issue,open,high,90,The driver called about the flat number. I hav...
3,7076,ORD-8476,2026-10-17T11:32,2026-10-17T13:52,south,Hyderabad,SwiftParcel,missing_parcel,in_progress,high,15,"Tracking says delivered, but I cannot find the..."
4,7060,ORD-8460,2026-10-14T14:04,2026-10-14T15:52,south,Hyderabad,RoutePost,damaged,resolved,normal,45,The seal was torn when I received the parcel.


## A. Explore the CSV

### 1. What came in the export?

Show the first five and last five rows, the shape, and the column names. Which column identifies an update's case? Which column tells you when that update happened?

In [126]:
df = raw_cases.copy()

print("First 5 rows:")
display(df.head())

print("Last 5 rows:")
display(df.tail())

print("Dataset Shape:", df.shape)

print("Column Names:")
print(df.columns.tolist())

First 5 rows:


,case_id,order_id,opened_at,updated_at,region,city,carrier,issue_type,status,priority,delay_minutes,customer_message
0,7011,ORD-8411,2026-10-06T13:17,2026-10-06T18:57,north,Delhi,ParcelBridge,missing_parcel,resolved,high,150,The parcel has had no scan since it left the s...
1,7027,ORD-8427,2026-10-09T10:44,2026-10-09T14:19,east,Kolkata,ParcelBridge,damaged,resolved,low,25,"The outer box arrived crushed, and one item wa..."
2,7033,ORD-8433,2026-10-10T10:31,2026-10-10T18:17,west,Pune,MetroDispatch,address_issue,open,high,90,The driver called about the flat number. I hav...
3,7076,ORD-8476,2026-10-17T11:32,2026-10-17T13:52,south,Hyderabad,SwiftParcel,missing_parcel,in_progress,high,15,"Tracking says delivered, but I cannot find the..."
4,7060,ORD-8460,2026-10-14T14:04,2026-10-14T15:52,south,Hyderabad,RoutePost,damaged,resolved,normal,45,The seal was torn when I received the parcel.


Last 5 rows:


,case_id,order_id,opened_at,updated_at,region,city,carrier,issue_type,status,priority,delay_minutes,customer_message
83,7056,ORD-8456,2026-10-14T09:23,2026-10-14T16:22,west,Ahmedabad,MetroDispatch,address_issue,open,normal,60,"Tracking shows ""address incomplete"". Please co..."
84,7006,ORD-8406,2026-10-05T14:08,2026-10-05T18:20,west,Pune,RoutePost,damaged,in-progress,high,-25,"The outer box arrived crushed, and one item wa..."
85,7022,ORD-8422,2026-10-08T11:47,2026-10-08T18:49,north,Jaipur,ParcelBridge,missing_parcel,in_progress,normal,60,The parcel has had no scan since it left the s...
86,7030,ORD-8430,2026-10-09T14:00,2026-10-09T20:52,east,Bhubaneswar,ParcelBridge,delayed,in_progress,low,25,I need the order before tomorrow. Can you chec...
87,7074,ORD-8474,2026-10-17T09:24,2026-10-17T16:19,south,Chennai,SwiftParcel,damaged,in_progress,normal,pending sync,"The outer box arrived crushed, and one item wa..."


Dataset Shape: (88, 12)
Column Names:
['case_id', 'order_id', 'opened_at', 'updated_at', 'region', 'city', 'carrier', 'issue_type', 'status', 'priority', 'delay_minutes', 'customer_message']


### Observations-
1. case_id identifies an update's case
2. updated_at tells when the update happened

### 2. How many cases are there?

Count the raw rows and the distinct case IDs. List the IDs that appear more than once and count how many updates each has. What do the two counts tell you about how the export was produced?

In [127]:
## total number of rows in the dataset
total_rows = len(df)
print("Total Rows:", total_rows)

## total number of unique cases in the dataset
unique_cases = df['case_id'].unique()
print("Unique Cases:", len(unique_cases))

## count of each unique case
case_counts = df['case_id'].value_counts()
print("Case Counts with ID's appearing more than once:", case_counts[case_counts > 1])



Total Rows: 88
Unique Cases: 80
Case Counts with ID's appearing more than once: case_id
7080    2
7015    2
7024    2
7008    2
7037    2
7049    2
7072    2
7060    2
Name: count, dtype: int64


### Observation
The total number of rows represents the number of records in the export, whereas the distinct case count represents the number of unique cases. Since the total rows are greater than the distinct case count, it indicates that some cases appear multiple times in the export. This suggests that the export contains multiple updates for the same case rather than just one record per case.

### 3. Where is the data incomplete?

Count missing values in each raw column. Then inspect `delay_minutes` and list examples of missing, invalid, and valid values. Which of those categories do blanks, `pending sync`, negative numbers, and zero belong to?

In [128]:
## missing value in each column
missing_values = df.isnull().sum()
print("Missing Values in Each Column:")
print(missing_values)

## rows with missing / blank / null values in 'delay_minutes' column
display(df[df["delay_minutes"].isnull()])

## unique values in 'delay_minutes' column
display(df["delay_minutes"].unique())

Missing Values in Each Column:
case_id             0
order_id            0
opened_at           0
updated_at          0
region              0
city                0
carrier             0
issue_type          0
status              0
priority            0
delay_minutes       1
customer_message    0
dtype: int64


,case_id,order_id,opened_at,updated_at,region,city,carrier,issue_type,status,priority,delay_minutes,customer_message
33,7003,ORD-8403,2026-10-05T10:45,2026-10-05T12:34,north,Delhi,SwiftParcel,delayed,open,normal,NaN,"The delivery window has passed, and tracking s..."


array(['150', '25', '90', '15', '45', '480', '240', '360', '0', '120',
       '60', '165', '420', '180', nan, 'pending sync', '-12', 'unknown',
       '225', '205', '390', '145', '-25'], dtype=object)

### Observation

| Value | Category | Reason |
|---|---|---|
| NaN / Blank | Missing | No value has been provided |
| pending sync | Invalid | Text instead of a numeric delay |
| unknown | Invalid | Delay value is unknown |
| Negative numbers (-12, -25) | Invalid | Delay cannot be negative |
| Zero (0) | Valid | Represents no delay |
| Positive numbers | Valid | Represents actual delay in minutes |

### 4. Which categories need standardization?

Show the distinct raw values of `status`, `region`, `carrier`, and `priority`. Find two labels that would be counted separately in a raw grouping even though they have the same meaning.

In [129]:
## unique values in 'status', 'region', 'carrier', and 'priority' columns
print(df["status"].unique())
print(df["region"].unique())
print(df["carrier"].unique())
print(df["priority"].unique())

['resolved' 'open' 'in_progress' 'In Progress' ' In Progress '
 'in-progress' 'RESOLVED' ' RESOLVED ' ' OPEN ']
['north' 'east' 'west' 'south' ' NORTH ']
['ParcelBridge' 'MetroDispatch' 'SwiftParcel' 'RoutePost'
 ' METRODISPATCH ']
['high' 'low' 'normal' 'urgent' ' HIGH ']


### Observation

All four columns contain inconsistent labels. The same category is represented using different capitalization, extra leading/trailing spaces, and different separators. As a result, when grouping or counting the raw values, these labels would be treated as separate categories even though they have the same meaning. These inconsistencies need standardization before performing accurate analysis.

### 5. What happened to case 7008?

Use `.loc` to display all updates for case 7008 with just the case ID, update time, status, and delay. Sort them by update time. Then use `.iloc` to select the last row of that result. Which status should a lookup tool return?

In [130]:
## Select a specific row by index using .loc
df_7008 = df.loc[df["case_id"] == 7008, ["case_id", "updated_at", "status", "delay_minutes"]].sort_values(by="updated_at", ascending=True)
print("Rows with case_id 7008:")
display(df_7008)

### Select last row by index using .iloc
df_last = df.iloc[-1]
print("Last Row:")
display(df_last)


Rows with case_id 7008:


,case_id,updated_at,status,delay_minutes
59,7008,2026-10-06T15:23,open,360
71,7008,2026-10-07T03:23,RESOLVED,390


Last Row:


case_id                                                          7074
order_id                                                     ORD-8474
opened_at                                            2026-10-17T09:24
updated_at                                           2026-10-17T16:19
region                                                          south
city                                                          Chennai
carrier                                                   SwiftParcel
issue_type                                                    damaged
status                                                    in_progress
priority                                                       normal
delay_minutes                                            pending sync
customer_message    The outer box arrived crushed, and one item wa...
Name: 87, dtype: object

## B. Clean and prepare the data

### 6. Can you normalize the text labels?

Create a copy and trim spaces and lowercase the region, carrier, issue type, status, and priority columns. Use a mapping to turn both `in progress` and `in-progress` into `in_progress`. Compare the distinct status labels before and after cleaning.

In [131]:
standardised_df = df.copy()

### Standardize the 'region', 'carrier', 'issue_type', 'status' and 'priority' columns to lowercase
standardised_df['region'] = standardised_df['region'].str.lower().str.strip()
standardised_df['carrier'] = standardised_df['carrier'].str.lower().str.strip()
standardised_df['issue_type'] = standardised_df['issue_type'].str.lower().str.strip()
standardised_df['status'] = standardised_df['status'].str.lower().str.strip()
standardised_df['priority'] = standardised_df['priority'].str.lower().str.strip()

standardised_df["status"] = standardised_df["status"].apply(lambda x: "in_progress" if x == "in progress" or x == "in-progress" else x)

print("Status values BEFORE cleaning:")
display(df["status"].unique())

print("Status counts BEFORE cleaning:")
display(df["status"].value_counts())

print("Total distinct statuses BEFORE cleaning:")
print(len(df["status"].unique()))


print("\nStatus values AFTER cleaning:")
display(standardised_df["status"].unique())

print("Status counts AFTER cleaning:")
display(standardised_df["status"].value_counts())

print("Total distinct statuses AFTER cleaning:")
print(len(standardised_df["status"].unique()))

Status values BEFORE cleaning:


array(['resolved', 'open', 'in_progress', 'In Progress', ' In Progress ',
       'in-progress', 'RESOLVED', ' RESOLVED ', ' OPEN '], dtype=object)

Status counts BEFORE cleaning:


status
resolved         34
open             27
in_progress      20
in-progress       2
In Progress       1
 In Progress      1
RESOLVED          1
 RESOLVED         1
 OPEN             1
Name: count, dtype: int64

Total distinct statuses BEFORE cleaning:
9

Status values AFTER cleaning:


array(['resolved', 'open', 'in_progress'], dtype=object)

Status counts AFTER cleaning:


status
resolved       36
open           28
in_progress    24
Name: count, dtype: int64

Total distinct statuses AFTER cleaning:
3


### 7. Can you turn a messy delay column into numbers?

Convert `delay_minutes` to numbers with pandas. Treat blanks, unparseable text, and negative values as missing. Keep zero as a valid measurement. How many measurements became missing? Inspect cases 7004, 7005, and 7006 to show the difference.

In [139]:
## Convert the 'delay_minutes' column to numeric, coercing errors to NaN
standardised_df["delay_minutes"] = pd.to_numeric(standardised_df["delay_minutes"], errors="coerce", downcast="integer")

##marking negative values as NaN
standardised_df.loc[standardised_df["delay_minutes"] < 0, "delay_minutes"] = np.nan

standardised_df["delay_minutes"].unique()

standardised_df.loc[standardised_df["case_id"].isin([7004, 7005, 7006]), ["case_id", "delay_minutes"]]

,case_id,delay_minutes
45,7005,NaN
55,7004,0.0
84,7006,NaN


### 8. How much time passed before each update?

Parse `opened_at` and `updated_at` as datetimes. Create a column giving the minutes between opening and that update. Find the three updates with the largest elapsed time and explain what this measure tells you about a case.

In [158]:
## parsing the 'opened_at' and 'updated_at' columns to datetime format, coercing errors to NaT
standardised_df["opened_at"] = pd.to_datetime(standardised_df["opened_at"], errors="coerce")
standardised_df["updated_at"] = pd.to_datetime(standardised_df["updated_at"], errors="coerce")

standardised_df["elapsed_minutes"] = (standardised_df["updated_at"] - standardised_df["opened_at"]).dt.total_seconds() / 60

standardised_df.sort_values(by="elapsed_minutes", ascending=False).head(3)

,case_id,order_id,opened_at,updated_at,region,city,carrier,issue_type,status,priority,delay_minutes,customer_message,elapsed_minutes
76,7080,ORD-8480,2026-10-18 09:33:00,2026-10-19 04:36:00,south,Chennai,parcelbridge,delayed,resolved,normal,180.0,I need the order before tomorrow. Can you chec...,1143.0
71,7008,ORD-8408,2026-10-06 09:11:00,2026-10-07 03:23:00,north,Lucknow,metrodispatch,delayed,resolved,normal,390.0,"The delivery window has passed, and tracking s...",1092.0
29,7037,ORD-8437,2026-10-11 08:08:00,2026-10-12 00:18:00,west,Pune,routepost,missing_parcel,open,urgent,420.0,The parcel has had no scan since it left the s...,970.0


## Observation
Elapsed time tells us how much time has passed between when a case was originally opened and when a particular update was recorded.

It helps us understand how long a case has been active before that update and identify cases that had updates much later than their opening time.

Also helps us identify cases which have status open for longest time without resolution yet

### 9. Which update should represent each case?

Sort by `updated_at` and keep the latest row for each `case_id`. The export is shuffled, so use the timestamp to decide which row to keep. Compare the status counts before and after deduplication.

In [169]:
## Get the latest cases based on updated_at timestamp
latest_cases = (
    standardised_df.
    sort_values(by="updated_at", ascending=True)
    .drop_duplicates(subset="case_id", keep="last")
)

display(latest_cases)

print("data set initially:", standardised_df["status"].value_counts())
print("latest_cases:", latest_cases["status"].value_counts())

,case_id,order_id,opened_at,updated_at,region,city,carrier,issue_type,status,priority,delay_minutes,customer_message,elapsed_minutes
79,7001,ORD-8401,2026-10-05 08:25:00,2026-10-05 08:46:00,north,Delhi,metrodispatch,delayed,open,high,120.0,"The delivery window has passed, and tracking s...",21.0
33,7003,ORD-8403,2026-10-05 10:45:00,2026-10-05 12:34:00,north,Delhi,swiftparcel,delayed,open,normal,NaN,"The delivery window has passed, and tracking s...",109.0
45,7005,ORD-8405,2026-10-05 12:50:00,2026-10-05 13:06:00,east,Kolkata,parcelbridge,missing_parcel,open,urgent,NaN,"Tracking says delivered, but I cannot find the...",16.0
6,7002,ORD-8402,2026-10-05 09:42:00,2026-10-05 14:42:00,north,Jaipur,routepost,delayed,in_progress,urgent,240.0,"The delivery window has passed, and tracking s...",300.0
84,7006,ORD-8406,2026-10-05 14:08:00,2026-10-05 18:20:00,west,Pune,routepost,damaged,in_progress,high,NaN,"The outer box arrived crushed, and one item wa...",252.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,7074,ORD-8474,2026-10-17 09:24:00,2026-10-17 16:19:00,south,Chennai,swiftparcel,damaged,in_progress,normal,NaN,"The outer box arrived crushed, and one item wa...",415.0
21,7078,ORD-8478,2026-10-17 14:01:00,2026-10-17 17:49:00,east,Kolkata,routepost,missing_parcel,open,normal,0.0,"Tracking says delivered, but I cannot find the...",228.0
19,7077,ORD-8477,2026-10-17 13:17:00,2026-10-17 18:03:00,south,Chennai,metrodispatch,address_issue,resolved,normal,480.0,The driver called about the flat number. I hav...,286.0
25,7079,ORD-8479,2026-10-18 08:04:00,2026-10-18 09:23:00,north,Lucknow,routepost,address_issue,resolved,low,480.0,"Tracking shows ""address incomplete"". Please co...",79.0


data set initially: status
resolved       36
open           28
in_progress    24
Name: count, dtype: int64
latest_cases: status
resolved       29
open           27
in_progress    24
Name: count, dtype: int64


### 10. Can you make the cleaning reusable?

Package the cleaning and deduplication into `prepare_cases(raw_cases: pd.DataFrame) -> pd.DataFrame`. Return a new table and leave the input intact. Print a short summary of how many rows came in, how many unique cases remained, and how many of those cases had unknown delays.

In [175]:
def prepare_cases(raw_cases: pd.DataFrame) -> pd.DataFrame:
    
    cleaned_df = raw_cases.copy()

    ### Standardize the 'region', 'carrier', 'issue_type', 'status' and 'priority' columns to lowercase
    cleaned_df['region'] = cleaned_df['region'].str.lower().str.strip()
    cleaned_df['carrier'] = cleaned_df['carrier'].str.lower().str.strip()
    cleaned_df['issue_type'] = cleaned_df['issue_type'].str.lower().str.strip()
    cleaned_df['status'] = cleaned_df['status'].str.lower().str.strip()
    cleaned_df['priority'] = cleaned_df['priority'].str.lower().str.strip()

    cleaned_df["status"] = cleaned_df["status"].apply(lambda x: "in_progress" if x == "in progress" or x == "in-progress" else x)

    ## Convert the 'delay_minutes' column to numeric, coercing errors to NaN
    cleaned_df["delay_minutes"] = pd.to_numeric(cleaned_df["delay_minutes"], errors="coerce", downcast="integer")

    ##marking negative values as NaN
    cleaned_df.loc[cleaned_df["delay_minutes"] < 0, "delay_minutes"] = np.nan

    ## parsing the 'opened_at' and 'updated_at' columns to datetime format, coercing errors to NaT
    cleaned_df["opened_at"] = pd.to_datetime(cleaned_df["opened_at"], errors="coerce")
    cleaned_df["updated_at"] = pd.to_datetime(cleaned_df["updated_at"], errors="coerce")

    cleaned_df["elapsed_minutes"] = (cleaned_df["updated_at"] - cleaned_df["opened_at"]).dt.total_seconds() / 60

    cleaned_df = (
        cleaned_df.
        sort_values(by="updated_at", ascending=True)
        .drop_duplicates(subset="case_id", keep="last")
    )

    unknown_delays = cleaned_df["delay_minutes"].isna().sum()

    print("Cases with unknown delays:", unknown_delays)

    return cleaned_df

In [178]:
print("Original rows:", len(df))
print("Original distinct cases:", df["case_id"].nunique())
print("Original cases with unknown delays:", df["delay_minutes"].isna().sum())

cleaned_df = prepare_cases(df)

print("Final rows:", len(cleaned_df))
print("Final distinct cases:", cleaned_df["case_id"].nunique())

display(cleaned_df.head())

Original rows: 88
Original distinct cases: 80
Original cases with unknown delays: 1
Cases with unknown delays: 9
Final rows: 80
Final distinct cases: 80


,case_id,order_id,opened_at,updated_at,region,city,carrier,issue_type,status,priority,delay_minutes,customer_message,elapsed_minutes
79,7001,ORD-8401,2026-10-05 08:25:00,2026-10-05 08:46:00,north,Delhi,metrodispatch,delayed,open,high,120.0,"The delivery window has passed, and tracking s...",21.0
33,7003,ORD-8403,2026-10-05 10:45:00,2026-10-05 12:34:00,north,Delhi,swiftparcel,delayed,open,normal,NaN,"The delivery window has passed, and tracking s...",109.0
45,7005,ORD-8405,2026-10-05 12:50:00,2026-10-05 13:06:00,east,Kolkata,parcelbridge,missing_parcel,open,urgent,NaN,"Tracking says delivered, but I cannot find the...",16.0
6,7002,ORD-8402,2026-10-05 09:42:00,2026-10-05 14:42:00,north,Jaipur,routepost,delayed,in_progress,urgent,240.0,"The delivery window has passed, and tracking s...",300.0
84,7006,ORD-8406,2026-10-05 14:08:00,2026-10-05 18:20:00,west,Pune,routepost,damaged,in_progress,high,NaN,"The outer box arrived crushed, and one item wa...",252.0


## C. Practise Python functions for agent tools

### 11. Can you validate a case ID from a request?

Write `parse_case_id(value: object) -> int`. Accept positive integers and digit strings after trimming spaces. Raise `ValueError` for other values, including booleans. Try it with `7001`, `" 7001 "`, `"ORD-8401"`, and `True`.

In [180]:
def parse_case_id(value: object) -> int:
    if isinstance(value, bool):
        raise ValueError(f"Invalid case_id value: {value}")
    else:
        if isinstance(value, int) and value > 0:
            return value
        elif isinstance(value, str):
            trimmed_value = value.strip()
            if trimmed_value.isdigit() and int(trimmed_value) > 0:
                return int(trimmed_value)
        raise ValueError(f"Invalid case_id value: {value}")


test_values = [7001, " 7001 ", "ORD-8401", True]

for value in test_values:
    try:
        parsed_value = parse_case_id(value)
        print(f"Parsed case_id from {value!r}: {parsed_value}")
    except ValueError as e:
        print(e)

Parsed case_id from 7001: 7001
Parsed case_id from ' 7001 ': 7001
Invalid case_id value: ORD-8401
Invalid case_id value: True


### 12. Can you look up the latest status?

Convert the cleaned table to a list of dictionaries. Write a lookup function that accepts a case ID and returns a dictionary with the case ID, status, carrier, and issue type. Return an error dictionary if the ID is missing from the dataset, and try a known ID and an absent one.

In [ ]:
def convert_to_dict(id: int) -> dict:
    if id not in cleaned_df["case_id"].values:
        return {
            "case_id": id,
            "status": None,
            "carrier": None,
            "issue_type": None,
        }
    return {
        "case_id": id,
        "status": cleaned_df.loc[cleaned_df["case_id"] == id, "status"].values[0],
        "carrier": cleaned_df.loc[cleaned_df["case_id"] == id, "carrier"].values[0],
        "issue_type": cleaned_df.loc[cleaned_df["case_id"] == id, "issue_type"].values[0],
    }

print(convert_to_dict(7001))
print(convert_to_dict(9999))

cases_list = cleaned_df.to_dict(orient="records")
print(cases_list[:3])

{'case_id': 7001, 'status': 'open', 'carrier': 'metrodispatch', 'issue_type': 'delayed'}
{'case_id': 9999, 'status': None, 'carrier': None, 'issue_type': None}
[{'case_id': 7001, 'order_id': 'ORD-8401', 'opened_at': Timestamp('2026-10-05 08:25:00'), 'updated_at': Timestamp('2026-10-05 08:46:00'), 'region': 'north', 'city': 'Delhi', 'carrier': 'metrodispatch', 'issue_type': 'delayed', 'status': 'open', 'priority': 'high', 'delay_minutes': 120.0, 'customer_message': 'The delivery window has passed, and tracking still shows the parcel in transit.', 'elapsed_minutes': 21.0}, {'case_id': 7003, 'order_id': 'ORD-8403', 'opened_at': Timestamp('2026-10-05 10:45:00'), 'updated_at': Timestamp('2026-10-05 12:34:00'), 'region': 'north', 'city': 'Delhi', 'carrier': 'swiftparcel', 'issue_type': 'delayed', 'status': 'open', 'priority': 'normal', 'delay_minutes': nan, 'customer_message': 'The delivery window has passed, and tracking still shows the parcel in transit.', 'elapsed_minutes': 109.0}, {'case

### 13. Which cases should the operations team review first?

Write a function that accepts the records, a region, and a delay threshold. Return active cases in that region with a known delay at or above the threshold. Use a lambda to order them by longest delay first, with smaller case IDs first when delays tie. Compare the North region at 120 and 240 minutes.

In [188]:
def get_priority_cases(records: list, region: str, delay_threshold: int = 120) -> list:
    cases = []
    for case in records:
        if case["region"] == region and case["delay_minutes"] >= delay_threshold and case["status"] != "resolved":
            cases.append(case)
    
    sorted_cases = sorted(cases, key=lambda x: (-x["delay_minutes"], x["case_id"]))
    
    return sorted_cases

get_priority_cases(cases_list, "north", 300)

[{'case_id': 7016,
  'order_id': 'ORD-8416',
  'opened_at': Timestamp('2026-10-07 12:02:00'),
  'updated_at': Timestamp('2026-10-07 15:30:00'),
  'region': 'north',
  'city': 'Jaipur',
  'carrier': 'swiftparcel',
  'issue_type': 'delayed',
  'status': 'open',
  'priority': 'normal',
  'delay_minutes': 480.0,
  'customer_message': 'I need the order before tomorrow. Can you check the latest scan?',
  'elapsed_minutes': 208.0},
 {'case_id': 7035,
  'order_id': 'ORD-8435',
  'opened_at': Timestamp('2026-10-10 12:46:00'),
  'updated_at': Timestamp('2026-10-10 16:14:00'),
  'region': 'north',
  'city': 'Lucknow',
  'carrier': 'routepost',
  'issue_type': 'damaged',
  'status': 'open',
  'priority': 'normal',
  'delay_minutes': 360.0,
  'customer_message': 'The outer box arrived crushed, and one item was broken.',
  'elapsed_minutes': 208.0}]

### 14. How can one run avoid reviewing a case twice?

Build a `DeliveryRun` class with a set on each instance. Give it methods to mark an ID as seen, check whether it was seen, and count distinct IDs. Create two instances, mark an ID twice in the first, and show what each instance remembers.

In [191]:
class DeliveryRun:
    def __init__(self):
        self.seen_cases = set()
    
    def mark_seen(self, case_id: int):
        self.seen_cases.add(case_id)
    
    def check_seen(self, case_id: int) -> bool:
        return case_id in self.seen_cases

    def count_distinct_seen_ids(self) -> int:
        return len(self.seen_cases)

run1 = DeliveryRun()
run2 = DeliveryRun()

run1.mark_seen(7001)
run1.mark_seen(7001)
run1.mark_seen(7002)

run2.mark_seen(7003)

print("Run 1:", run1.seen_cases)
print("Run 2:", run2.seen_cases)

print("Run 1 has seen 7001:", run1.check_seen(7001))
print("Run 2 has seen 7001:", run2.check_seen(7001))

print("Run 1 distinct IDs:", run1.count_distinct_seen_ids())
print("Run 2 distinct IDs:", run2.count_distinct_seen_ids())

Run 1: {7001, 7002}
Run 2: {7003}
Run 1 has seen 7001: True
Run 2 has seen 7001: False
Run 1 distinct IDs: 2
Run 2 distinct IDs: 1


### 15. Can you turn case records into short descriptions?

Write a named function that accepts one case and returns a line such as `Case 7008 | north | resolved`. Apply it to a list of records using Python's built-in `map()`. Use `functools.reduce()` to total the known delays for active cases and compare the result with `sum()`.

In [196]:
def format_case(case: dict) -> str:
    return f"Case {case['case_id']} | {case['region']} | {case['status']}"

formatted_cases = list(map(format_case, cases_list))

print(cases_list[:5])
print(formatted_cases[:5])

[{'case_id': 7001, 'order_id': 'ORD-8401', 'opened_at': Timestamp('2026-10-05 08:25:00'), 'updated_at': Timestamp('2026-10-05 08:46:00'), 'region': 'north', 'city': 'Delhi', 'carrier': 'metrodispatch', 'issue_type': 'delayed', 'status': 'open', 'priority': 'high', 'delay_minutes': 120.0, 'customer_message': 'The delivery window has passed, and tracking still shows the parcel in transit.', 'elapsed_minutes': 21.0}, {'case_id': 7003, 'order_id': 'ORD-8403', 'opened_at': Timestamp('2026-10-05 10:45:00'), 'updated_at': Timestamp('2026-10-05 12:34:00'), 'region': 'north', 'city': 'Delhi', 'carrier': 'swiftparcel', 'issue_type': 'delayed', 'status': 'open', 'priority': 'normal', 'delay_minutes': nan, 'customer_message': 'The delivery window has passed, and tracking still shows the parcel in transit.', 'elapsed_minutes': 109.0}, {'case_id': 7005, 'order_id': 'ORD-8405', 'opened_at': Timestamp('2026-10-05 12:50:00'), 'updated_at': Timestamp('2026-10-05 13:06:00'), 'region': 'east', 'city': 'Ko

In [201]:
from functools import reduce

total_delay = reduce(
    lambda acc, case: acc + case["delay_minutes"] 
    if pd.notna(case["delay_minutes"]) and case["status"] != "resolved" 
    else acc, 
    cases_list, 
    0
)

total_delay_sum = sum(
    case["delay_minutes"]
    for case in cases_list
    if pd.notna(case["delay_minutes"]) and case["status"] != "resolved"
)

print("Total delay for unresolved cases:", total_delay)
print("Total delay for unresolved cases (using sum):", total_delay_sum)

Total delay for unresolved cases: 5570.0
Total delay for unresolved cases (using sum): 5570.0


## D. Analyze the backlog with NumPy and pandas

### 16. How many delays meet each threshold?

Build a NumPy array of known delays from active cases. Use boolean masks to count the delays at or above 60, 120, and 240 minutes. Also report the minimum, median, and maximum. How many active cases were excluded because their delays were unknown?

### 17. What happens when you combine two conditions?

Use `.loc` to find high or urgent active cases with a known delay of at least 120 minutes. Show only the case ID, region, carrier, priority, and delay. Sort the result and use `.iloc` to display the first three matching rows.

### 18. Which carriers have the most active cases?

Group active cases by carrier. For each carrier, show the number of active cases, the number with a known delay, and the average known delay. Why might `.size()` and `.count()` give different numbers?

### 19. Can you assign a label to each known delay?

Use `Series.map()` with a dictionary to give each priority a readable label. Then use `Series.apply()` with a named function to label delays as `unknown`, `under 120 minutes`, or `120 minutes or more`. Show the counts for each delay label among active cases.

### 20. Which region and carrier combinations need attention?

Build a grouped table by region and carrier for active cases. Show the active count and the number at or above 120 minutes. Calculate the percentage meeting the threshold **among known delays** and use a missing result when a group has no known delays. Which combinations look different when you compare counts with percentages?

## E. Choose and explain a chart

### 21. Which carrier has the most active cases?

Make a Seaborn count plot using active cases by carrier. Add labels above the bars and compare them with your grouped counts. Give the chart a title and label both axes.

In [189]:
import seaborn as sns
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


### 22. How are known delays distributed?

Plot a histogram of known delays for active cases. Draw a line at 120 minutes, then compare the chart with the exact count from a mask. How could changing the histogram bins change what you notice?

### 23. Which regions have a wider spread of delays?

Make a Seaborn box plot of known active delays by region. Compare medians and spread. Choose one region with a point beyond a whisker and look up the case or cases that may explain it.

### 24. How did new exceptions change over time?

Count unique cases by the calendar day on which they opened. Plot the counts in date order as a line chart. Identify the busiest day. Explain why cases opened on a day and cases still active on that day are different measures.

### 25. Do longer recorded delays come with longer update times?

On the cleaned table, make a scatter plot of delay minutes against minutes from opening to the latest update. Use region as the color and exclude unknown delays. Describe one pattern or exception, and name another factor you would want to check before drawing a conclusion.

## F. Build Python tools and reports

### 26. Can your program exchange a tool request as JSON?

Start with `{"tool": "get_case_status", "case_id": "7008"}` as JSON text. Parse it, call the lookup tool, and convert the returned dictionary back to JSON text. Decide what error the application should return for malformed JSON, an unsupported tool, and an absent case. Demonstrate one request for each outcome.

### 27. Can you wrap an HTTP request in a Python function?

Use the class practice endpoint at `https://jsonplaceholder.typicode.com/todos/1`. Write a function that accepts a task ID, sends a GET request with a timeout, checks the response status, and returns a dictionary with the ID, title, and completion state. Decide how to return a useful error if the request cannot complete. You can use a sample response dictionary when practising offline.

### 28. Can you save and reload a daily report?

Build a dictionary with the total active cases, unknown delays, counts by carrier, and number of active cases delayed at least 120 minutes. Save it as `practice_delivery_report.json` using UTF-8. Reload it and show that the values match. Convert any pandas or NumPy values to JSON-compatible Python types where needed.

### 29. Can your tool validate a delay search request?

Start with `{"tool": "find_delayed_cases", "region": " North ", "min_delay": 120}` as JSON text. Normalize the region and check that it exists in the cleaned table. Accept a non-negative whole number for the threshold. Then reuse your delay search function and return the matching case IDs and count in a dictionary. Return a clear error for an unknown region or invalid threshold.

### 30. Can you build a daily delivery report from a CSV?

Write a function that takes a CSV path and a delay threshold. Load and clean the data, then return a dictionary with the number of active cases, how many have unknown delays, counts by carrier, and how many known delays meet the threshold. Use an f-string to write a one-sentence briefing from these values. Handle a missing file with a clear message. Run the report for 120 and 240 minutes and explain what changed.